# Cliniguard Model Comparison

This notebook loads the two trained models (LightGBM and Logistic Regression) and compares them on the validation set. It reports overall accuracy, RED‑class recall, and visualises feature importance (LightGBM) vs coefficients (Logistic Regression).


In [ ]:
import os, joblib, numpy as np, pandas as pd, math, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, recall_score
from sklearn.preprocessing import StandardScaler


In [ ]:
BASE_DIR = os.path.abspath(os.path.dirname(__file__))
DATA_PATH = os.path.join(BASE_DIR, 'cliniguard_all_datasets.csv')
MODEL_LGB = os.path.join(BASE_DIR, 'cliniguard_model.joblib')
MODEL_LR = os.path.join(BASE_DIR, 'cliniguard_lr_model.joblib')
SCALER_PATH = os.path.join(BASE_DIR, 'cliniguard_scaler.joblib')


In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)[['question','answer','label']]
# Feature engineering (same as training)
DRUG_TERMS = {"mg","dose","dosage","tablet","capsule","injection","oral","iv","intravenous","amoxicillin","ibuprofen","metformin","insulin","aspirin","atorvastatin","omeprazole","paracetamol","acetaminophen","warfarin","morphine","prednisone","antibiotic","medication","drug","prescribe","contraindication","side","effect","adverse"}
CONTEXT_TERMS = {"patient","allergy","allergic","age","weight","pediatric","adult","vital","history","medication","diagnosis","symptom","report","female","male","chronic","acute","clinical","contraindication"}
UNCERTAIN_WORDS = {"maybe","possibly","might","could","uncertain","clear","unknown","approximately","seems","appears","suggest","perhaps","likely","probably","assume","think","believe","estimate","roughly","sometimes","often"}

def tokenize(t):
    return t.lower().split() if isinstance(t, str) else []
def med_isp(text):
    w = tokenize(text)
    if not w: return 1.0
    hits = sum(1 for x in w if any(t in x for t in DRUG_TERMS))
    return round(1.0 - min(hits / max(len(w)*0.05, 1), 1.0), 4
def c_aas(text):
    w = tokenize(text)
    if not w: return 1.0
    hits = sum(1 for x in w if any(t in x for t in CONTEXT_TERMS))
    return round(1.0 - min(hits / max(len(w)*0.04, 1), 1.0), 4
def med_eem(text):
    w = tokenize(text); n = len(w)
    if n == 0: return 0.0
    p = sum(1 for x in w if any(t in x for t in UNCERTAIN_WORDS)) / n
    eps = 1e-9
    H = -(p*math.log2(p+eps) + (1-p)*math.log2(1-p+eps))
    return round(min(H*(1+p), 1.0), 4)
def cdt(answer, question):
    """Cosine‑drift‑type similarity between answer and question.

    Returns a value in the range [0, 1] where higher values indicate a larger
    semantic drift (more dissimilar). This implementation is fully JSON‑safe and
    contains no stray control characters.
    """
    def wvec(t: str) -> dict:
        """Convert a string to a word‑frequency dictionary."""
        f = {}
        for x in tokenize(t):
            f[x] = f.get(x, 0) + 1
        return f

    # Build term‑frequency vectors for the question and the answer
    v1, v2 = wvec(question), wvec(answer)

    # Union of all words appearing in either string
    vocab = set(v1) | set(v2)
    if not vocab:
        return 0.5          # fallback for empty inputs

    # Cosine similarity components
    dot = sum(v1.get(x, 0) * v2.get(x, 0) for x in vocab)
    m1 = math.sqrt(sum(x ** 2 for x in v1.values()))
    m2 = math.sqrt(sum(x ** 2 for x in v2.values()))
    if m1 == 0 or m2 == 0:
        return 0.5          # avoid division‑by‑zero

    # Convert similarity to drift (higher = more drift) and round
    return round(1.0 - dot / (m1 * m2), 4)
# Compute feature matrix
signals = []
for _, row in df.iterrows():
    ans = row['answer']
    q = row['question']
    sig = [med_isp(ans), c_aas(ans), med_eem(ans), cdt(ans, q)]
    signals.append(sig)
X = np.array(signals)
y = df['label'].astype(int).values


In [ ]:
# Load scaler and transform
scaler = joblib.load(SCALER_PATH)
X_scaled = scaler.transform(X)


In [ ]:
# Load models
lgb_model = joblib.load(MODEL_LGB)
lr_model = joblib.load(MODEL_LR)


In [ ]:
# Simple train/val split (same seed as training)
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
def evaluate(name, model):
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    # RED class is label 2
    red_recall = recall_score(y_val, preds, labels=[2], average='macro')
    return {'model': name, 'accuracy': acc, 'red_recall': red_recall}

results = []
results.append(evaluate('LightGBM', lgb_model))
results.append(evaluate('LogisticRegression', lr_model))

import pandas as pd
df_res = pd.DataFrame(results)
display(df_res)


In [ ]:
# Feature importance (LightGBM)
importances = lgb_model.feature_importances_
feat_names = ['med_isp','c_aas','med_eem','cdt']
sns.barplot(x=importances, y=feat_names, orient='h')
plt.title('LightGBM Feature Importance')
plt.show()


In [ ]:
# Coefficients (Logistic Regression)
coeffs = lr_model.coef_[0]  # shape (n_classes, n_features) – we take the RED class (index 2)
# The model was trained with class_weight='balanced', so coefficients are comparable
sns.barplot(x=coeffs, y=feat_names, orient='h')
plt.title('Logistic Regression Coefficients (RED class)')
plt.show()
